# 🧬 Notebook 3B — Large Custom DNA Transformer Across All Tokenizers

## One architecture. Five DNA representations. Multi-GPU training.

Notebook 2 taught us how to **build a Transformer from scratch** and how DNA can be represented in several ways.

Notebook 3B scales that idea up.

We will train a **fresh larger custom Transformer** for each tokenizer:

```text
1. Single nucleotide
2. One-hot
3. Overlapping 6-mer
4. Non-overlapping 6-mer
5. BPE
```

Every run uses the same cleaned dataset, train/validation split, architecture, optimizer settings, epoch count, BF16 precision, and DDP strategy.

The main variable is therefore:

> **How we represent the DNA sequence.**

### How to read this notebook

| Marker | Meaning |
|---|---|
| 🔒 **RUN ONLY** | Infrastructure. Run it; you do not need to memorize it. |
| 👀 **READ** | Important code. Read the comments and follow the main idea. |
| 🧠 **BUILD IT** | A core concept turned into code. Read this one closely. |
| ✏️ **EDIT ME** | Change a value, re-run, and see what moves. |
| 🔲 **YOUR TURN** | A line is left blank on purpose. Write it, then run the check cell below it. |
| ✅ **CHECKPOINT** | Stop and answer before moving on. |

Every notebook in this bootcamp uses these same six markers.

## 🧭 Python survival guide — read this once, then come back when needed

You do **not** need to memorize Python syntax. When you see unfamiliar code, first identify the job it is doing.

| Python word | Plain-English meaning | Tiny example |
|---|---|---|
| **variable** | A name that stores a value | `k = 6` |
| **function** | A reusable mini-program that performs one job | `gc_content(sequence)` |
| **argument** | A value you give to a function | `gc_content("ACGT")` |
| **return** | The value a function gives back | `return gc_fraction` |
| **list** | An ordered collection | `["A", "C", "G", "T"]` |
| **dictionary (`dict`)** | Named values stored as key → value pairs | `{"A": 1, "C": 2}` |
| **DataFrame** | A pandas table: rows are examples, columns are properties | `df.head()` |
| **boolean mask** | A True/False filter that selects rows | `df[df["label"] == 1]` |
| **class** | A blueprint for an object that stores data and behavior together | `class DNASet(...)` |
| **method** | A function that belongs to an object/class | `model.forward(...)` |

### How to read a function

```python
def gc_content(sequence):      # function name + input
    gc = ...                   # work done inside the function
    return gc                  # value sent back
```

Read that as:

> “Given a `sequence`, calculate something called `gc`, then give `gc` back.”

### How to read a class

```python
class ExampleModel(nn.Module):
    def __init__(self):
        ...

    def forward(self, x):
        ...
```

- `__init__` = **what pieces does this object contain?**
- `forward` = **what happens to the input when it moves through the model?**
- `self` = **this particular object**. You normally do not pass it yourself.

Whenever a cell is marked **🔒 RUN ONLY**, focus on the explanation above it rather than every Python detail.



## 🧭 HPC survival guide — the words you will keep seeing

| Term | Plain-English meaning |
|---|---|
| **Slurm** | The scheduler that decides when and where your job runs on Perlmutter. |
| **job** | One request sent to Slurm: hardware + time + commands to execute. |
| **node** | One compute machine. A Perlmutter GPU node contains multiple GPUs. |
| **GPU** | The accelerator doing most of the neural-network math. |
| **QOS** | A Slurm queue/policy that controls what resources a job may request. |
| **reservation** | Compute nodes held for a specific event/time window, such as the bootcamp. |
| **DDP** | *DistributedDataParallel*: multiple Python processes cooperate to train one model. |
| **rank** | The ID of one DDP process: `0`, `1`, `2`, ... |
| **local rank** | Which GPU a process uses on its current node. |
| **world size** | Total number of DDP processes/GPUs participating in the run. |
| **all-reduce** | A communication step that combines gradients from all ranks so every model copy stays synchronized. |
| **BF16** | A 16-bit number format that uses less memory and can run efficiently on A100 GPUs. |
| **throughput** | How much work the system completes per second, e.g. examples/second. |

### Mental model for DDP

```text
rank 0 → GPU 0 → different training examples ┐
rank 1 → GPU 1 → different training examples ├─ all-reduce gradients → synchronized model
rank 2 → GPU 2 → different training examples ┤
rank 3 → GPU 3 → different training examples ┘
```

The important idea is **one model training run**, not four unrelated models.


## Learning objectives

By the end of Notebook 3B, you should be able to:

1. Reuse the five tokenization methods from Notebook 2.
2. Explain why token count changes Transformer compute.
3. Scale a custom Transformer to a larger architecture.
4. Train that architecture with multiple GPUs using DDP.
5. Use BF16 mixed precision on the GPU.
6. Train **one fresh model per tokenizer**.
7. Compare tokenizers using:
   - validation AUROC,
   - AUPRC,
   - training time,
   - examples/second,
   - peak GPU memory,
   - parameter count,
   - mean token count.
8. Decide which tokenizer gives the best balance between biological performance and computational cost.

# 1. Connection to Notebook 2

Notebook 2 used the following representations:

| Tokenizer | Representation | Approximate tokens for 200 bp |
|---|---|---:|
| Single nucleotide | A, C, G, T as IDs | 200 |
| One-hot | one 4D vector per base | 200 |
| Overlapping 6-mer | sliding 6-base words | 195 |
| Non-overlapping 6-mer | chunks of 6 bases | 33 |
| BPE | learned variable-length DNA chunks | variable |

We keep the same important rules:

- PAD ID = `0` for discrete tokenizers.
- Single nucleotide real IDs do not collide with PAD.
- One-hot uses 4 dimensions: A/C/G/T.
- Overlapping 6-mers use all sliding windows.
- Non-overlapping 6-mers discard the final incomplete remainder in this implementation.
- BPE is learned from **training DNA only**.
- BPE uses PAD = 0, UNK = 1, real tokens beginning at 2.
- Dynamic padding pads only to the longest sequence **in the current batch**.

# 2. Why Token Count Matters More for a Larger Transformer

Self-attention compares token positions with other token positions.

A useful conceptual proxy is:

```text
attention work ∝ token_count²
```

For example:

```text
200 tokens → about 40,000 pairwise positions
 33 tokens → about  1,089 pairwise positions
```

This is **not an exact runtime formula**, but it explains why tokenizer choice can dramatically change the compute required by attention.

Notebook 3B lets us measure that consequence on the GPU.

# 3. Setup

In [ ]:
import os
import sys
import json
import time
import shlex
import subprocess
import py_compile

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# All bootcamp notebooks resolve the SAME folder, so a dataset built in
# Notebook 0 is found by Notebooks 1-3 even if you launch them from
# elsewhere. Override by setting the DNA_BOOTCAMP_HOME environment variable.
PROJECT_DIR = Path(os.environ.get("DNA_BOOTCAMP_HOME", ".")).expanduser().resolve()
print("📁 Project directory:", PROJECT_DIR)

DATA_DIR = Path(
    "/global/cfs/cdirs/m4388/projects/project7/ctcf_k562_example"
)

RESULTS_DIR = (PROJECT_DIR / "notebook3b_results")

SCRIPTS_DIR = (PROJECT_DIR / "notebook3b_scripts")

SLURM_LOG_DIR = (RESULTS_DIR / "slurm_logs")

for directory in [RESULTS_DIR, SCRIPTS_DIR, SLURM_LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

NOTEBOOK_PYTHON = sys.executable

print("Notebook Python:", NOTEBOOK_PYTHON)
print("Data directory:", DATA_DIR)
print("Results directory:", RESULTS_DIR)


def project_relative(path):
    """Express a path relative to PROJECT_DIR for use inside a Slurm script.

    Slurm splits #SBATCH directives on whitespace, so an absolute path that
    contains a space — '/.../Untitled Folder/logs/job-%j.out' — is read as a
    directive plus garbage, and sbatch rejects the script. Quoting does not
    save you: shlex.quote uses single quotes, and the srun body is already
    inside bash -c '...'.

    Jobs are submitted with PROJECT_DIR as the working directory, so the
    relative form points at exactly the same file.
    """
    path = Path(path).resolve()
    try:
        return path.relative_to(PROJECT_DIR).as_posix()
    except ValueError:
        return str(path)


if " " in str(PROJECT_DIR):
    print()
    print("⚠️  Your project directory contains a space:")
    print(f"      {PROJECT_DIR}")
    print("   Generated job scripts therefore use project-relative paths,")
    print("   and jobs are submitted from that directory. This works — but")
    print("   renaming the folder to remove spaces is safer, because most")
    print("   HPC tooling has the same problem in places we do not control.")


# 4. Large-Model Configuration

Notebook 2 used a deliberately small teaching model.

Notebook 3B uses a substantially larger model while keeping the architecture recognizable.

The default is:

```text
embedding dimension = 256
attention heads     = 8
Transformer layers  = 8
feed-forward width  = 4 × embedding dimension
```

Every parameter in the custom model is trainable because the model starts from scratch.

## Throughput-first batch strategy

Like Notebook 3A, we specify a **local batch per GPU**.

For example:

```text
2 GPUs × 32 examples/GPU = global batch 64
```

This gives each GPU a meaningful amount of work.

In [ ]:
# ✏️ EDIT ME — where you are running, and the architecture to train

NERSC_ACCOUNT = "m4388"

# "shared"   -> testing outside bootcamp hours (1-2 GPUs, no reservation)
# "bootcamp" -> during the bootcamp (uses that day's reservation)
RUN_MODE = "bootcamp"
BOOTCAMP_DAY = 1

# shared: 1 or 2.  bootcamp: any multiple of 4, or 1-4 on a single node.
GPU_COUNT = 4

# Same local batch for every tokenizer, so the comparison stays fair.
LOCAL_BATCH_SIZE = 32

# This is much larger than the Notebook 2 model. The goal is to find out
# whether a from-scratch Transformer can reach DNABERT's Notebook 3A score.
MODEL_CONFIG = {
    "n_embd": 256,
    "n_head": 8,
    "n_layer": 8,
    "dropout": 0.10,
    "epochs": 10,
    "local_batch_size": LOCAL_BATCH_SIZE,
    "learning_rate": 3e-4,
    "weight_decay": 0.01,
    "warmup_ratio": 0.10,
    "precision": "bf16",
    "bpe_merges": 80,
    "seed": 42,
}

TOKENIZER_ORDER = ["single_nucleotide", "one_hot", "overlap_6mer",
                   "nonoverlap_6mer", "bpe"]

pd.Series(MODEL_CONFIG, name="value")

## Where are you running this?

A Slurm job needs to know two different things, and students mix them up
constantly:

1. **How many GPUs do you want?** — that is `GPU_COUNT`.
2. **Which pool of machines are you allowed to take them from?** — that is
   the QOS, and during the bootcamp, a *reservation*.

A Perlmutter GPU node has **4 A100 GPUs**. So asking for 8 GPUs is really
asking for 2 whole nodes. The cell below does that arithmetic for you.

| `RUN_MODE` | When | QOS | GPUs you can ask for |
|---|---|---|---|
| `"shared"` | Testing, outside bootcamp hours | `shared` | 1 or 2, on one shared node |
| `"bootcamp"` | During the bootcamp | `regular` + that day's reservation | whole nodes, up to the day's limit |

**The bootcamp reservations:**

| Day | Reservation | Window | Nodes | GPUs |
|---|---|---|---:|---:|
| 1 | `bootcamp_day1` | 11:00 – 22:00 | 20 | 80 |
| 2 | `bootcamp_day2` | 11:00 – 22:00 | 30 | 120 |
| 3 | `bootcamp_day3` | 11:00 – 22:00 | 30 | 120 |
| 4 | `bootcamp_day4` | 08:00 – 00:00 | 40 | 160 |
| 5 | `bootcamp_day5` | 08:00 – 11:00 | 30 | 120 |

A reservation is **not** extra hardware you are entitled to on top of the
queue — it is a block of nodes held aside so your job does not wait behind
everyone else's. Outside the window, the reservation does not exist and the
job will be rejected.

In [ ]:
# 🔒 RUN ONLY — turn "I want N GPUs" into Slurm flags that actually get you N

import math

GPUS_PER_NODE = 4          # a Perlmutter GPU node has 4 A100s

# name, nodes held, window
BOOTCAMP_RESERVATIONS = {
    1: ("bootcamp_day1", 20, "11:00-22:00"),
    2: ("bootcamp_day2", 30, "11:00-22:00"),
    3: ("bootcamp_day3", 30, "11:00-22:00"),
    4: ("bootcamp_day4", 40, "08:00-00:00"),
    5: ("bootcamp_day5", 30, "08:00-11:00"),
}


def resolve_slurm(gpu_count, run_mode=None, day=None):
    """Return the Slurm settings needed to obtain `gpu_count` GPUs."""
    run_mode = RUN_MODE if run_mode is None else run_mode
    day = BOOTCAMP_DAY if day is None else day
    gpu_count = int(gpu_count)

    if gpu_count < 1:
        raise ValueError("Request at least one GPU.")

    if run_mode == "shared":
        if gpu_count > 2:
            raise ValueError(
                f"RUN_MODE='shared' is for testing and allows 1-2 GPUs, "
                f"but GPU_COUNT={gpu_count}. During the bootcamp set "
                f"RUN_MODE='bootcamp' and BOOTCAMP_DAY."
            )
        return {"qos": "shared", "reservation": None, "nodes": 1,
                "tasks_per_node": gpu_count, "gpus_per_node": gpu_count,
                "gpus": gpu_count, "window": "any"}

    if run_mode == "bootcamp":
        if day not in BOOTCAMP_RESERVATIONS:
            raise ValueError(f"BOOTCAMP_DAY must be one of "
                             f"{sorted(BOOTCAMP_RESERVATIONS)}, got {day}.")
        name, max_nodes, window = BOOTCAMP_RESERVATIONS[day]
        nodes = math.ceil(gpu_count / GPUS_PER_NODE)

        if nodes > 1 and gpu_count % GPUS_PER_NODE != 0:
            raise ValueError(
                f"A multi-node run must use whole nodes: GPU_COUNT must be a "
                f"multiple of {GPUS_PER_NODE}, got {gpu_count}. Try "
                f"{nodes * GPUS_PER_NODE}."
            )
        if nodes > max_nodes:
            raise ValueError(
                f"{gpu_count} GPUs needs {nodes} nodes, but {name} only holds "
                f"{max_nodes} ({max_nodes * GPUS_PER_NODE} GPUs)."
            )

        per_node = gpu_count if nodes == 1 else GPUS_PER_NODE
        return {"qos": "regular", "reservation": name, "nodes": nodes,
                "tasks_per_node": per_node, "gpus_per_node": per_node,
                "gpus": gpu_count, "window": window}

    raise ValueError("RUN_MODE must be 'shared' or 'bootcamp'.")


if MODEL_CONFIG["n_embd"] % MODEL_CONFIG["n_head"] != 0:
    raise ValueError("n_embd must be divisible by n_head.")

PLAN = resolve_slurm(GPU_COUNT)

print("Slurm plan for", GPU_COUNT, "GPU(s)")
print("-" * 42)
for key in ["qos", "reservation", "nodes", "tasks_per_node",
            "gpus_per_node", "window"]:
    print(f"  {key:<15}: {PLAN[key]}")
print()
print(f"  {PLAN['nodes']} node(s) x {PLAN['gpus_per_node']} GPU(s) "
      f"= {PLAN['gpus']} GPU(s) total")
if PLAN["reservation"]:
    print(f"  ⚠️  {PLAN['reservation']} only exists during {PLAN['window']}.")

# 5. The Larger Custom Transformer

We preserve the architecture students built in Notebook 2:

```text
input representation
↓
token embedding OR one-hot linear projection
↓
learned positional embedding
↓
8 pre-LayerNorm Transformer blocks
↓
final LayerNorm
↓
masked mean pooling
↓
binary classifier
```

Each block still contains:

```text
multi-head self-attention
+
feed-forward network
+
residual connections
+
LayerNorm
```

The model is larger, but the logic is the same.

## 🧭 Model-language refresher

| Term | Meaning in this custom Transformer |
|---|---|
| `n_embd` | Width of each token vector. `256` means every token is represented by 256 numbers inside the Transformer. |
| `n_head` | Number of attention heads operating in parallel inside each block. |
| `n_layer` | Number of Transformer blocks stacked one after another. |
| `head_size` | Features handled by one attention head: `n_embd / n_head`. |
| **logits** | Final raw class scores before probabilities. |
| **masked mean pooling** | Average the real token vectors while ignoring PAD positions. |
| **from scratch** | Model weights start random; unlike DNABERT, there is no pretrained knowledge. |


## Attention scaling remains correct

For each attention head:

```python
scale = head_size ** -0.5
```

We scale by the **head dimension**, not the total embedding dimension.

This is the same correctness rule used in Notebook 2.

# 6. GPU Strategy

Each tokenizer is trained as a **separate fresh model**, but the five models are trained sequentially inside one GPU allocation.

For one tokenizer:

```text
rank 0 / GPU 0 ─┐
                 ├─ DDP gradient all-reduce → one model
rank 1 / GPU 1 ─┘
```

Then that model is saved and released from memory before the next tokenizer begins.

This avoids waiting in the queue five separate times while still giving each tokenizer run the full requested multi-GPU allocation.

## BF16

All five models use BF16 autocast for compatible GPU operations.

The model parameters themselves remain standard PyTorch trainable parameters; autocast chooses lower precision for eligible operations during forward computation.

# 7. Write the Shared DDP Helper

**📄 `notebook3b_scripts/ddp_common.py`** — this is the real program Slurm will run on the GPU nodes.

It is written to disk by the `%%writefile` magic below, so what you read here
is exactly what executes. Edit the cell and re-run it to change the job.

## 🧩 Function map — shared DDP helpers

Notebook 3B uses the same distributed-training ideas as 3A:

| Function | Plain-English job |
|---|---|
| `setup_distributed` | Gives each rank a GPU and connects all ranks. |
| `cleanup_distributed` | Closes DDP communication cleanly. |
| `reduce_training_stats` | Combines training counts/loss across GPUs. |
| `binary_metrics` | Converts predictions into the classification metrics used in the final comparison. |

If this feels familiar from 3A, that is intentional: **the HPC framework is the same; the model changes.**


In [ ]:
%%writefile notebook3b_scripts/ddp_common.py
import os

import numpy as np
import torch
import torch.distributed as dist

from sklearn.metrics import (confusion_matrix, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score)


def setup_distributed():
    rank = int(os.environ.get("RANK", "0"))
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    world_size = int(os.environ.get("WORLD_SIZE", "1"))

    distributed = world_size > 1
    visible_gpu_count = torch.cuda.device_count()

    print(f"[DDP setup] RANK={rank} | " f"LOCAL_RANK={local_rank} | "
        f"WORLD_SIZE={world_size} | " f"visible_GPUs={visible_gpu_count} | "
        f"CUDA_VISIBLE_DEVICES="
        f"{os.environ.get('CUDA_VISIBLE_DEVICES', '<not set>')}", flush=True)

    if visible_gpu_count < world_size:
        raise RuntimeError(
            f"Rank {rank} sees only {visible_gpu_count} GPU(s), "
            f"but WORLD_SIZE={world_size}.")

    if local_rank >= visible_gpu_count:
        raise RuntimeError(f"LOCAL_RANK={local_rank}, but only "
            f"{visible_gpu_count} CUDA device ordinal(s) are visible.")

    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    print(f"[GPU mapping] rank={rank} → cuda:{local_rank} | "
        f"{torch.cuda.get_device_name(local_rank)}", flush=True)

    if distributed:
        dist.init_process_group(
            backend="nccl",
            init_method="env://",
            rank=rank,
            world_size=world_size,
            device_id=device,
        )

    return (distributed, rank, world_size, local_rank, device)


def cleanup_distributed(distributed):
    if distributed and dist.is_initialized():
        dist.destroy_process_group()


def reduce_training_stats(loss_sum, correct, n, device, distributed):
    values = torch.tensor([loss_sum, correct, n], dtype=torch.float64,
        device=device)

    if distributed:
        dist.all_reduce(values, op=dist.ReduceOp.SUM)

    loss_total, correct_total, n_total = values.tolist()

    return (loss_total / n_total, correct_total / n_total, int(n_total))


def binary_metrics(y_true, y_pred, scores):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {"accuracy": (tp + tn) / (tp + tn + fp + fn),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "specificity": (tn / (tn + fp) if (tn + fp) else np.nan),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auroc": roc_auc_score(y_true, scores),
        "auprc": average_precision_score(y_true, scores),
        "true_negative": int(tn), "false_positive": int(fp),
        "false_negative": int(fn), "true_positive": int(tp)}

In [ ]:
# 🔒 RUN ONLY — confirm the file landed, and bind it to a variable

# %%writefile above wrote to a path relative to the notebook's working
# directory. SCRIPTS_DIR is the absolute form of that same folder — this
# check fails loudly if the two ever disagree, instead of writing the
# script somewhere the Slurm job will not find it.
DDP_COMMON = SCRIPTS_DIR / "ddp_common.py"

if not DDP_COMMON.exists():
    raise FileNotFoundError(
        f"Expected {DDP_COMMON} after running the %%writefile cell above.\n"
        f"The notebook's working directory is {Path.cwd()}, and %%writefile "
        f"wrote to 'notebook3b_scripts/ddp_common.py' relative to it.\n"
        f"Run the %%writefile cell, or start Jupyter from {PROJECT_DIR}."
    )

py_compile.compile(str(DDP_COMMON), doraise=True)

print("✅", DDP_COMMON)
print(f"   {len(DDP_COMMON.read_text().splitlines()):,} lines, valid Python")


# 8. Write the Large Custom-Transformer Training Program

**📄 `notebook3b_scripts/custom_all_tokenizers.py`** — this is the real program Slurm will run on the GPU nodes.

It is written to disk by the `%%writefile` magic below, so what you read here
is exactly what executes. Edit the cell and re-run it to change the job.

## 🧩 Map of the large training script

This is the longest cell in Notebook 3B. Do **not** read it top-to-bottom as one giant program. Use this map:

| Layer of the program | Important names | Purpose |
|---|---|---|
| **Data** | `clean_split` | Make the same clean train/validation split for every tokenizer. |
| **Tokenization** | `tokenize_single_nucleotide`, `tokenize_one_hot`, `tokenize_overlap_6mer`, `tokenize_nonoverlap_6mer`, `train_bpe`, `apply_bpe`, `tokenize_bpe` | Recreate the five representation methods from Notebook 2. |
| **Batching** | `PreTokenizedDataset`, `make_collate_fn` | Store tokens and dynamically pad one batch at a time. |
| **Transformer pieces** | `AttentionHead`, `MultiHeadAttention`, `FeedForward`, `TransformerBlock` | The same building blocks you constructed in Notebook 2. |
| **Whole model** | `DNATransformer` | Assemble embeddings/projection + positions + blocks + pooling + classifier. |
| **Optimization** | `build_optimizer`, `train_epoch`, `evaluate` | Update parameters and measure validation performance. |
| **One tokenizer run** | `train_one_tokenizer` | Train/save one fresh model for one tokenizer. |
| **Whole experiment** | `main` | Loop through all five tokenizers and create the combined comparison table. |

### Important reading rule

When you see:

```python
def train_one_tokenizer(...):
```

read it as:

> “Everything required to train **one fresh model** for **one representation**.”

Then `main()` repeats that same recipe for all five methods.


In [ ]:
%%writefile notebook3b_scripts/custom_all_tokenizers.py
import argparse
import collections
import itertools
import json
import os
import time
from pathlib import Path

# Keep student-facing logs focused on training results.
# Library errors remain visible; routine advisory messages are suppressed.
import warnings

from huggingface_hub import logging as hf_hub_logging
from transformers.utils import logging as transformers_logging

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module=r"torch\.distributed\.c10d_logger",
)

hf_hub_logging.set_verbosity_error()
transformers_logging.set_verbosity_error()

import numpy as np
import pandas as pd
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F

from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

from transformers import get_linear_schedule_with_warmup

from ddp_common import (setup_distributed, cleanup_distributed,
    reduce_training_stats, binary_metrics)


PAD_ID = 0
SINGLE_NUC_VOCAB = {"A": 1, "C": 2, "G": 3, "T": 4}
SINGLE_NUC_VOCAB_SIZE = 5
ONE_HOT_DIM = 4
K = 6

KMER_VOCAB = {"".join(chars): i + 1 for i, chars in enumerate(
        itertools.product("ACGT", repeat=K))}
KMER_VOCAB_SIZE = len(KMER_VOCAB) + 1

TOKENIZER_ORDER = ["single_nucleotide", "one_hot", "overlap_6mer",
    "nonoverlap_6mer", "bpe"]


def clean_split(data_dir, seed):
    data_dir = Path(data_dir)

    seqs = [line.strip().upper() for line in open(data_dir / "seqs.txt")
        if line.strip()]

    labels = [int(line.strip()) for line in open(data_dir / "labels.txt")
        if line.strip()]

    df = pd.DataFrame({"sequence": seqs, "label": labels})

    df["length"] = df["sequence"].str.len()
    expected_length = int(df["length"].mode().iloc[0])

    valid = (df["length"].eq(expected_length) & df["sequence"].apply(
            lambda seq: set(seq) <= set("ACGT")))

    conflicts = set(df.groupby("sequence")["label"] .nunique()
        .loc[lambda values: values > 1] .index)

    clean_df = (df[valid & ~df["sequence"].isin(conflicts)]
        .drop_duplicates("sequence") .reset_index(drop=True))

    train_df, val_df = train_test_split(clean_df, test_size=0.20,
        random_state=seed, stratify=clean_df["label"])

    return (clean_df, train_df.reset_index(drop=True),
        val_df.reset_index(drop=True))


def tokenize_single_nucleotide(seq):
    return [SINGLE_NUC_VOCAB[base] for base in seq]


def tokenize_one_hot(seq):
    mapping = {"A": [1.0, 0.0, 0.0, 0.0], "C": [0.0, 1.0, 0.0, 0.0],
        "G": [0.0, 0.0, 1.0, 0.0], "T": [0.0, 0.0, 0.0, 1.0]}

    return [mapping[base] for base in seq]


def tokenize_overlap_6mer(seq):
    return [KMER_VOCAB[seq[i:i+K]] for i in range(len(seq) - K + 1)]


def tokenize_nonoverlap_6mer(seq):
    usable_length = (len(seq) // K) * K

    return [KMER_VOCAB[seq[i:i+K]] for i in range(0, usable_length, K)]


def train_bpe(sequences, num_merges=80):
    corpus = [list(seq) for seq in sequences]
    rules = []

    for _ in range(num_merges):
        pair_counts = collections.Counter()

        for tokens in corpus:
            pair_counts.update(zip(tokens[:-1], tokens[1:]))

        if not pair_counts:
            break

        best_pair = pair_counts.most_common(1)[0][0]
        merged_token = "".join(best_pair)
        new_corpus = []

        for tokens in corpus:
            new_tokens = []
            i = 0

            while i < len(tokens):
                if (i < len(tokens) - 1 and tokens[i] == best_pair[0]
                    and tokens[i + 1] == best_pair[1]):
                    new_tokens.append(merged_token)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1

            new_corpus.append(new_tokens)

        corpus = new_corpus
        rules.append(best_pair)

    return rules


def apply_bpe(seq, rules):
    tokens = list(seq)

    for pair in rules:
        merged_token = "".join(pair)
        new_tokens = []
        i = 0

        while i < len(tokens):
            if (i < len(tokens) - 1 and tokens[i] == pair[0]
                and tokens[i + 1] == pair[1]):
                new_tokens.append(merged_token)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1

        tokens = new_tokens

    return tokens


def build_tokenizer_info(name, train_sequences, bpe_merges):
    if name == "single_nucleotide":
        return {"name": name, "fn": tokenize_single_nucleotide,
            "continuous": False, "vocab_size": SINGLE_NUC_VOCAB_SIZE,
            "input_dim": None, "position_capacity": 200}

    if name == "one_hot":
        return {"name": name, "fn": tokenize_one_hot, "continuous": True,
            "vocab_size": None, "input_dim": ONE_HOT_DIM,
            "position_capacity": 200}

    if name == "overlap_6mer":
        return {"name": name, "fn": tokenize_overlap_6mer, "continuous": False,
            "vocab_size": KMER_VOCAB_SIZE, "input_dim": None,
            "position_capacity": 195}

    if name == "nonoverlap_6mer":
        return {"name": name, "fn": tokenize_nonoverlap_6mer,
            "continuous": False, "vocab_size": KMER_VOCAB_SIZE,
            "input_dim": None, "position_capacity": 33}

    if name == "bpe":
        rules = train_bpe(train_sequences, num_merges=bpe_merges)

        tokens_seen = set()

        for seq in train_sequences:
            tokens_seen.update(apply_bpe(seq, rules))

        vocab = {token: i + 2 for i, token in enumerate(sorted(tokens_seen))}

        def tokenize_bpe(seq):
            return [vocab.get(token, 1) for token in apply_bpe(seq, rules)]

        return {"name": name, "fn": tokenize_bpe, "continuous": False,
            "vocab_size": len(vocab) + 2, "input_dim": None,
            "position_capacity": 200, "bpe_rules": rules, "bpe_vocab": vocab}

    raise ValueError(f"Unknown tokenizer: {name}")


class PreTokenizedDataset(Dataset):
    def __init__(self, seqs, labels, tokenizer_info):
        self.continuous = tokenizer_info["continuous"]
        self.inputs = []

        for seq in seqs:
            tokens = tokenizer_info["fn"](seq)

            if self.continuous:
                tensor = torch.tensor(tokens, dtype=torch.float32)
            else:
                tensor = torch.tensor(tokens, dtype=torch.long)

            self.inputs.append(tensor)

        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {"input": self.inputs[idx], "label": self.labels[idx]}


def make_collate_fn(tokenizer_info):
    continuous = tokenizer_info["continuous"]
    input_dim = tokenizer_info["input_dim"]

    def collate(batch):
        lengths = [item["input"].shape[0] for item in batch]

        max_len = max(lengths)
        batch_size = len(batch)

        if continuous:
            x_batch = torch.zeros(batch_size, max_len, input_dim,
                dtype=torch.float32)
        else:
            x_batch = torch.full((batch_size, max_len), PAD_ID,
                dtype=torch.long)

        attention_mask = torch.zeros(batch_size, max_len, dtype=torch.long)

        labels = torch.zeros(batch_size, dtype=torch.long)

        for i, item in enumerate(batch):
            length = item["input"].shape[0]
            x_batch[i, :length] = item["input"]
            attention_mask[i, :length] = 1
            labels[i] = item["label"]

        return {"input": x_batch, "attention_mask": attention_mask,
            "label": labels}

    return collate


class AttentionHead(nn.Module):
    def __init__(self, n_embd, head_size, dropout):
        super().__init__()

        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.scale = head_size ** -0.5

    def forward(self, x, attention_mask):
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        scores = q @ k.transpose(-2, -1)
        scores = scores * self.scale

        key_mask = attention_mask.unsqueeze(1)
        scores = scores.masked_fill(key_mask == 0, float("-inf"))

        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)

        return weights @ v


class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()

        assert n_embd % n_head == 0
        head_size = n_embd // n_head

        self.heads = nn.ModuleList([AttentionHead(n_embd, head_size, dropout)
            for _ in range(n_head)])

        self.projection = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attention_mask):
        combined = torch.cat([head(x, attention_mask) for head in self.heads],
            dim=-1)

        return self.dropout(self.projection(combined))


class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()

        self.net = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()

        self.attention = MultiHeadAttention(n_embd, n_head, dropout)
        self.feed_forward = FeedForward(n_embd, dropout)
        self.norm1 = nn.LayerNorm(n_embd)
        self.norm2 = nn.LayerNorm(n_embd)

    def forward(self, x, attention_mask):
        x = x + self.attention(self.norm1(x), attention_mask)

        x = x + self.feed_forward(self.norm2(x))

        return x


class DNATransformer(nn.Module):
    def __init__(self, position_capacity, n_embd, n_head, n_layer, dropout,
        vocab_size=None, input_dim=None):
        super().__init__()

        assert (vocab_size is None) != (input_dim is None)

        self.continuous = input_dim is not None

        if self.continuous:
            self.token_embedding = nn.Linear(input_dim, n_embd)
        else:
            self.token_embedding = nn.Embedding(vocab_size, n_embd,
                padding_idx=PAD_ID)

        self.position_embedding = nn.Embedding(position_capacity, n_embd)

        self.blocks = nn.ModuleList([TransformerBlock(n_embd, n_head, dropout)
            for _ in range(n_layer)])

        self.final_norm = nn.LayerNorm(n_embd)
        self.classifier = nn.Linear(n_embd, 2)

    def forward(self, x, attention_mask):
        _, token_count = x.shape[:2]

        token_vectors = self.token_embedding(x)

        positions = torch.arange(token_count, device=x.device)

        h = token_vectors + self.position_embedding(positions)

        for block in self.blocks:
            h = block(h, attention_mask)

        h = self.final_norm(h)

        mask = attention_mask.unsqueeze(-1).float()
        pooled = ((h * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1))

        return self.classifier(pooled)


def build_optimizer(model, learning_rate, weight_decay):
    try:
        optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate,
            weight_decay=weight_decay, fused=True)
        mode = "AdamW fused=True"
    except (TypeError, RuntimeError):
        optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate,
            weight_decay=weight_decay)
        mode = "AdamW standard"

    return optimizer, mode


def train_epoch(model, optimizer, scheduler, loader, sampler, epoch, device,
    distributed):
    model.train()

    if sampler is not None:
        sampler.set_epoch(epoch)

    loss_sum = 0.0
    correct = 0
    n = 0

    for batch_index, batch in enumerate(loader):
        x = batch["input"].to(device, non_blocking=True)
        mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits = model(x, mask)
            loss = F.cross_entropy(logits, labels)

        loss.backward()

        if epoch == 1 and batch_index == 0:
            missing_grads = [name
                for name, parameter in model.named_parameters()
                if parameter.requires_grad and parameter.grad is None]

            if missing_grads:
                raise RuntimeError("Trainable parameters without gradients: "
                    + ", ".join(missing_grads))

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        predictions = logits.argmax(dim=1)

        loss_sum += loss.item() * len(labels)
        correct += (predictions == labels).sum().item()
        n += len(labels)

    return reduce_training_stats(loss_sum, correct, n, device, distributed)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()

    loss_sum = 0.0
    correct = 0
    n = 0
    scores = []
    predictions = []
    truth = []

    for batch in loader:
        x = batch["input"].to(device, non_blocking=True)
        mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits = model(x, mask)
            loss = F.cross_entropy(logits, labels)

        pred = logits.argmax(dim=1)
        prob = torch.softmax(logits.float(), dim=1)[:, 1]

        loss_sum += loss.item() * len(labels)
        correct += (pred == labels).sum().item()
        n += len(labels)

        scores.extend(prob.cpu().numpy())
        predictions.extend(pred.cpu().numpy())
        truth.extend(labels.cpu().numpy())

    return {"loss": loss_sum / n, "accuracy": correct / n,
        "scores": np.asarray(scores), "predictions": np.asarray(predictions),
        "true": np.asarray(truth)}


def train_one_tokenizer(tokenizer_name, clean_df, train_df, val_df, args,
    distributed, rank, world_size, device):
    if distributed:
        dist.barrier()

    if rank == 0:
        print("\n" + "=" * 88, flush=True)
        print(f"TOKENIZER: {tokenizer_name}", flush=True)
        print("=" * 88, flush=True)

    preprocessing_start = time.time()

    info = build_tokenizer_info(tokenizer_name, train_df["sequence"].tolist(),
        args.bpe_merges)

    train_dataset = PreTokenizedDataset(train_df["sequence"].tolist(),
        train_df["label"].tolist(), info)

    val_dataset = PreTokenizedDataset(val_df["sequence"].tolist(),
        val_df["label"].tolist(), info)

    token_lengths = np.asarray([len(info["fn"](seq))
        for seq in clean_df["sequence"]])

    preprocessing_seconds = time.time() - preprocessing_start

    train_sampler = None

    if distributed:
        train_sampler = DistributedSampler(train_dataset,
            num_replicas=world_size, rank=rank, shuffle=True, seed=args.seed)

    collate_fn = make_collate_fn(info)

    # Inputs are already pre-tokenized in memory.
    # For this small dataset, extra DataLoader worker processes
    # add overhead instead of useful preprocessing throughput.
    worker_count = 0

    train_loader = DataLoader(train_dataset, batch_size=args.local_batch_size,
        shuffle=train_sampler is None, sampler=train_sampler,
        collate_fn=collate_fn, pin_memory=True, num_workers=worker_count,
        persistent_workers=worker_count > 0)

    val_loader = DataLoader(val_dataset, batch_size=args.local_batch_size,
        shuffle=False, collate_fn=collate_fn, pin_memory=True,
        num_workers=worker_count, persistent_workers=worker_count > 0)

    # Same initial seed for the same architecture on every rank.
    torch.manual_seed(args.seed)
    torch.cuda.manual_seed_all(args.seed)

    model = DNATransformer(position_capacity=info["position_capacity"],
        n_embd=args.n_embd, n_head=args.n_head, n_layer=args.n_layer,
        dropout=args.dropout, vocab_size=info["vocab_size"],
        input_dim=info["input_dim"]).to(device)

    total_parameters = sum(p.numel() for p in model.parameters())

    trainable_parameters = sum(p.numel() for p in model.parameters()
        if p.requires_grad)

    if trainable_parameters != total_parameters:
        raise RuntimeError("Custom Transformer should train all parameters.")

    if distributed:
        model = DDP(model, device_ids=[device.index],
            output_device=device.index, gradient_as_bucket_view=True)

    optimizer, optimizer_mode = build_optimizer(model, args.learning_rate,
        args.weight_decay)

    total_steps = len(train_loader) * args.epochs
    warmup_steps = int(total_steps * args.warmup_ratio)

    scheduler = get_linear_schedule_with_warmup(optimizer,
        num_warmup_steps=warmup_steps, num_training_steps=total_steps)

    # Independent dropout streams after DDP model synchronization.
    torch.manual_seed(args.seed + rank)
    torch.cuda.manual_seed_all(args.seed + rank)

    if rank == 0:
        print(f"GPUs: {world_size}", flush=True)
        print(f"Precision: {args.precision}", flush=True)
        print(f"Local batch/GPU: {args.local_batch_size}", flush=True)
        print(f"Global batch: {args.local_batch_size * world_size}",
            flush=True)
        print(f"Mean token count: {token_lengths.mean():.1f}", flush=True)
        print(f"Parameters: {total_parameters:,}", flush=True)
        print(f"Optimizer: {optimizer_mode}", flush=True)

    if distributed:
        dist.barrier()

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(device)
    torch.cuda.synchronize(device)

    training_start = time.time()
    processed_examples = 0
    history = []
    final_val = None

    # Keep the BEST epoch for THIS tokenizer, not the last one.
    best_auroc = -1.0
    best_val = None
    best_epoch = 0
    best_state = None

    for epoch in range(1, args.epochs + 1):
        epoch_start = time.time()

        train_loss, train_accuracy, examples_seen = train_epoch(model,
            optimizer, scheduler, train_loader, train_sampler, epoch, device,
            distributed)

        processed_examples += examples_seen

        if distributed:
            dist.barrier()

        if rank == 0:
            eval_model = model.module if distributed else model

            final_val = evaluate(eval_model, val_loader, device)

            val_auroc = roc_auc_score(final_val["true"], final_val["scores"])
            val_auprc = average_precision_score(final_val["true"],
                final_val["scores"])

            epoch_seconds = time.time() - epoch_start

            history.append({"epoch": epoch, "train_loss": train_loss,
                "train_accuracy": train_accuracy,
                "val_loss": final_val["loss"],
                "val_accuracy": final_val["accuracy"], "val_auroc": val_auroc,
                "val_auprc": val_auprc, "epoch_seconds": epoch_seconds,
                "learning_rate": optimizer.param_groups[0]["lr"]})

            if val_auroc > best_auroc:
                best_auroc = val_auroc
                best_val = final_val
                best_epoch = epoch
                best_state = {k: v.detach().cpu().clone()
                              for k, v in eval_model.state_dict().items()}

            if final_val["loss"] > 0.69 and val_auroc < 0.55:
                print("  \U0001F6A8 Collapsed to chance. Lower "
                      "--learning_rate.", flush=True)

            print(f"Epoch {epoch}/{args.epochs} | "
                f"train loss={train_loss:.4f} | "
                f"train acc={train_accuracy:.3f} | "
                f"val loss={final_val['loss']:.4f} | "
                f"AUROC={val_auroc:.4f} | " f"AUPRC={val_auprc:.4f} | "
                f"{epoch_seconds:.1f}s", flush=True)

        if distributed:
            dist.barrier()

    torch.cuda.synchronize(device)
    training_time = time.time() - training_start

    local_peak_bytes = torch.cuda.max_memory_allocated(device)
    peak_tensor = torch.tensor([float(local_peak_bytes)], dtype=torch.float64,
        device=device)

    if distributed:
        dist.all_reduce(peak_tensor, op=dist.ReduceOp.MAX)

    max_peak_bytes = peak_tensor.item()
    total_memory_bytes = torch.cuda.get_device_properties(device).total_memory

    result = None

    if rank == 0:
        history_df = pd.DataFrame(history)
        best_row = history_df.loc[history_df["val_auroc"].idxmax()]

        # Restore this tokenizer's best epoch before reporting.
        eval_model = model.module if distributed else model

        if best_state is not None:
            eval_model.load_state_dict(best_state)
            final_val = best_val
            print(f"\u21A9\uFE0F  Restored weights from epoch {best_epoch} "
                  f"(AUROC {best_auroc:.4f}).", flush=True)

        final_metrics = binary_metrics(final_val["true"],
            final_val["predictions"], final_val["scores"])

        predictions_df = val_df[["sequence", "label"]].copy()
        predictions_df["predicted_label"] = final_val["predictions"]
        predictions_df["binding_probability"] = final_val["scores"]
        predictions_df["correct"] = (predictions_df["label"]
            == predictions_df["predicted_label"])

        examples_per_second = processed_examples / training_time

        peak_memory_gb = max_peak_bytes / (1024 ** 3)
        total_memory_gb = total_memory_bytes / (1024 ** 3)
        peak_memory_percent = 100.0 * max_peak_bytes / total_memory_bytes

        run_name = f"custom_large_{tokenizer_name}"
        output_dir = Path(args.output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        prefix = output_dir / run_name

        history_df.to_csv(str(prefix) + "_history.csv", index=False)
        predictions_df.to_csv(str(prefix) + "_predictions.csv", index=False)

        eval_model = model.module if distributed else model
        checkpoint_path = str(prefix) + "_final_checkpoint.pt"

        torch.save({"tokenizer": tokenizer_name,
                "state_dict": eval_model.state_dict(),
                "position_capacity": info["position_capacity"],
                "vocab_size": info["vocab_size"],
                "input_dim": info["input_dim"], "n_embd": args.n_embd,
                "n_head": args.n_head, "n_layer": args.n_layer,
                "dropout": args.dropout, "bpe_rules": info.get("bpe_rules"),
                "bpe_vocab": info.get("bpe_vocab")}, checkpoint_path)

        result = {"run_name": run_name, "family": "custom_transformer",
            "tokenizer": tokenizer_name, "strategy": "large_scratch_ddp",
            "num_gpus": int(world_size), "precision": args.precision,
            "n_embd": int(args.n_embd), "n_head": int(args.n_head),
            "n_layer": int(args.n_layer), "dropout": float(args.dropout),
            "epochs": int(args.epochs),
            "local_batch_size": int(args.local_batch_size),
            "global_batch_size": int(args.local_batch_size * world_size),
            "learning_rate": float(args.learning_rate),
            "weight_decay": float(args.weight_decay),
            "warmup_ratio": float(args.warmup_ratio),
            "random_seed": int(args.seed), "bpe_merges": int(args.bpe_merges),
            "vocab_size": (int(info["vocab_size"])
                if info["vocab_size"] is not None else None), "input_dim": (
                int(info["input_dim"]) if info["input_dim"] is not None
                else None),
            "position_capacity": int(info["position_capacity"]),
            "mean_token_count": float(token_lengths.mean()),
            "median_token_count": float(np.median(token_lengths)),
            "min_token_count": int(token_lengths.min()),
            "max_token_count": int(token_lengths.max()),
            "trainable_parameters": int(trainable_parameters),
            "total_parameters": int(total_parameters),
            "best_val_auroc": float(best_row["val_auroc"]),
            "best_epoch": int(best_row["epoch"]),
            "final_val_accuracy": float(final_metrics["accuracy"]),
            "final_val_precision": float(final_metrics["precision"]),
            "final_val_recall": float(final_metrics["recall"]),
            "final_val_specificity": float(final_metrics["specificity"]),
            "final_val_f1": float(final_metrics["f1"]),
            "final_val_auroc": float(final_metrics["auroc"]),
            "final_val_auprc": float(final_metrics["auprc"]),
            "training_time_seconds": float(training_time),
            "examples_per_second": float(examples_per_second),
            "preprocessing_seconds": float(preprocessing_seconds),
            "peak_gpu_memory_gb": float(peak_memory_gb),
            "gpu_memory_capacity_gb": float(total_memory_gb),
            "peak_gpu_memory_percent": float(peak_memory_percent),
            "optimizer": optimizer_mode, "checkpoint_path": checkpoint_path}

        with open(str(prefix) + "_summary.json", "w") as handle:
            json.dump(result, handle, indent=2)

        print("\nFINAL TOKENIZER REPORT", flush=True)
        print(
            f"{tokenizer_name} | best AUROC={result['best_val_auroc']:.4f} | "
            f"time={training_time:.1f}s | "
            f"examples/s={examples_per_second:.1f} | "
            f"peak memory={peak_memory_gb:.2f} GB", flush=True)

    if distributed:
        dist.barrier()

    # Release this model before creating the next tokenizer model.
    del model
    del optimizer
    del scheduler
    del train_loader
    del val_loader
    del train_dataset
    del val_dataset

    torch.cuda.empty_cache()

    if distributed:
        dist.barrier()

    return result


def main():
    parser = argparse.ArgumentParser()

    parser.add_argument("--data_dir", required=True)
    parser.add_argument("--output_dir", required=True)
    parser.add_argument("--n_embd", type=int, required=True)
    parser.add_argument("--n_head", type=int, required=True)
    parser.add_argument("--n_layer", type=int, required=True)
    parser.add_argument("--dropout", type=float, required=True)
    parser.add_argument("--epochs", type=int, required=True)
    parser.add_argument("--local_batch_size", type=int, required=True)
    parser.add_argument("--learning_rate", type=float, required=True)
    parser.add_argument("--weight_decay", type=float, required=True)
    parser.add_argument("--warmup_ratio", type=float, required=True)
    parser.add_argument("--precision", choices=["bf16"], default="bf16")
    parser.add_argument("--bpe_merges", type=int, default=80)
    parser.add_argument("--seed", type=int, default=42)

    args = parser.parse_args()

    if args.n_embd % args.n_head != 0:
        raise ValueError("n_embd must be divisible by n_head")

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU required")

    if not torch.cuda.is_bf16_supported():
        raise RuntimeError("Notebook 3B expects BF16-capable GPUs")

    (distributed, rank, world_size, local_rank, device) = setup_distributed()

    try:
        np.random.seed(args.seed)
        torch.manual_seed(args.seed)
        torch.cuda.manual_seed_all(args.seed)
        torch.set_float32_matmul_precision("high")

        clean_df, train_df, val_df = clean_split(args.data_dir, args.seed)

        if rank == 0:
            print("\n=== NOTEBOOK 3B LARGE CUSTOM TRANSFORMER ===", flush=True)
            print(f"Clean sequences: {len(clean_df)}", flush=True)
            print(f"Train: {len(train_df)} | Validation: {len(val_df)}", flush=True)
            print(f"GPUs: {world_size}", flush=True)
            print(f"Architecture: n_embd={args.n_embd}, "
                f"n_head={args.n_head}, n_layer={args.n_layer}", flush=True)
            print(f"Local batch/GPU: {args.local_batch_size}", flush=True)
            print(f"Global batch: {args.local_batch_size * world_size}", flush=True)
            print(f"Tokenizers: {TOKENIZER_ORDER}", flush=True)

        combined_results = []

        for tokenizer_name in TOKENIZER_ORDER:
            result = train_one_tokenizer(tokenizer_name, clean_df, train_df,
                val_df, args, distributed, rank, world_size, device)

            if rank == 0:
                combined_results.append(result)

        if rank == 0:
            combined_df = pd.DataFrame(combined_results)
            combined_path = Path(args.output_dir) / "custom_all_tokenizers_summary.csv"
            combined_df.to_csv(combined_path, index=False)

            print("\n" + "=" * 88, flush=True)
            print("ALL TOKENIZERS COMPLETED", flush=True)
            print("=" * 88, flush=True)
            print(combined_df[["tokenizer", "mean_token_count",
                        "total_parameters", "best_val_auroc",
                        "training_time_seconds", "examples_per_second",
                        "peak_gpu_memory_gb"]].to_string(index=False),
                flush=True)
            print(f"\nSaved combined table: {combined_path}", flush=True)

    finally:
        cleanup_distributed(distributed)


if __name__ == "__main__":
    main()

In [ ]:
# 🔒 RUN ONLY — confirm the file landed, and bind it to a variable

# %%writefile above wrote to a path relative to the notebook's working
# directory. SCRIPTS_DIR is the absolute form of that same folder — this
# check fails loudly if the two ever disagree, instead of writing the
# script somewhere the Slurm job will not find it.
TRAIN_SCRIPT = SCRIPTS_DIR / "custom_all_tokenizers.py"

if not TRAIN_SCRIPT.exists():
    raise FileNotFoundError(
        f"Expected {TRAIN_SCRIPT} after running the %%writefile cell above.\n"
        f"The notebook's working directory is {Path.cwd()}, and %%writefile "
        f"wrote to 'notebook3b_scripts/custom_all_tokenizers.py' relative to it.\n"
        f"Run the %%writefile cell, or start Jupyter from {PROJECT_DIR}."
    )

py_compile.compile(str(TRAIN_SCRIPT), doraise=True)

print("✅", TRAIN_SCRIPT)
print(f"   {len(TRAIN_SCRIPT.read_text().splitlines()):,} lines, valid Python")


## What the training program does

The single distributed job performs:

```text
create identical cleaned split
↓
train single-nucleotide model
↓
save results + checkpoint
↓
release GPU memory
↓
train one-hot model
↓
save
↓
overlapping 6-mer
↓
non-overlapping 6-mer
↓
BPE
↓
combined comparison table
```

Every tokenizer receives a **fresh randomly initialized Transformer**.

No tokenizer inherits weights from another tokenizer.

For the BPE model, the saved checkpoint also contains the learned **BPE merge rules and BPE vocabulary**, so the representation can be reconstructed later.

# 9. What Is Held Constant?

For all five tokenizer runs:

```text
same cleaned DNA sequences
same 80/20 stratified split
same random seed
same embedding dimension
same attention-head count
same Transformer-layer count
same dropout
same local batch/GPU
same optimizer family
same learning rate
same epoch count
same BF16 precision
same GPU count
```

That is important because it makes tokenizer representation the main experimental variable.

### One unavoidable parameter-count difference

Discrete tokenizers use an embedding table whose size depends on vocabulary size.

Therefore total parameter count can differ slightly between tokenizers even though the Transformer core is identical.

That difference is real and should be reported rather than hidden.

# 10. SLURM Helpers

## 🧩 Slurm helper functions

These functions are infrastructure, not Transformer architecture:

- `validate_shell_script(...)` → checks the generated shell script before submission.
- `submit_and_stream(...)` → submits with `sbatch`, streams the output, watches job state, and reports final `sacct` information.

If a student understands **what job is being submitted** and **what resources were requested**, they do not need to memorize the subprocess code.


In [ ]:
# 🔒 RUN ONLY — shell validation + job monitoring

def validate_shell_script(path):
    result = subprocess.run(["bash", "-n", str(path)], capture_output=True,
        text=True)

    if result.returncode != 0:
        print(result.stderr)
        return False

    print("✅ Shell syntax valid:", path.name)
    return True


def submit_and_stream(script_path, job_name, poll_seconds=2.0):
    # A NERSC Jupyter session may itself have a CUDA mask.
    # Do not pass that mask into this new batch allocation.
    submit_env = os.environ.copy()

    inherited_gpu_env = {name: submit_env.get(name) for name in [
            "CUDA_VISIBLE_DEVICES", "NVIDIA_VISIBLE_DEVICES",
            "ROCR_VISIBLE_DEVICES", "GPU_DEVICE_ORDINAL"]
        if name in submit_env}

    for name in ["CUDA_VISIBLE_DEVICES", "NVIDIA_VISIBLE_DEVICES",
        "ROCR_VISIBLE_DEVICES", "GPU_DEVICE_ORDINAL"]:
        submit_env.pop(name, None)

    print("Notebook GPU environment:",
        inherited_gpu_env if inherited_gpu_env else "<none>")
    print("Submitting with inherited GPU visibility removed.")

    submit = subprocess.run(["sbatch", str(script_path)],
        capture_output=True, text=True, env=submit_env,
        cwd=str(PROJECT_DIR))

    if submit.returncode != 0:
        print(submit.stderr)
        raise RuntimeError("sbatch rejected the job")

    job_id = submit.stdout.strip().split()[-1]
    output_file = SLURM_LOG_DIR / f"{job_name}-{job_id}.out"

    print("Submitted job", job_id)
    print("Output:", output_file)

    last_size = 0

    while True:
        if output_file.exists():
            with output_file.open("r") as handle:
                handle.seek(last_size)
                text = handle.read()
                if text:
                    print(text, end="")
                last_size = handle.tell()

        active = subprocess.run(["squeue", "-h", "-j", job_id],
            capture_output=True, text=True).stdout.strip()

        if not active:
            if output_file.exists():
                with output_file.open("r") as handle:
                    handle.seek(last_size)
                    text = handle.read()
                    if text:
                        print(text, end="")
            break

        time.sleep(poll_seconds)

    summary = subprocess.run(["sacct", "-j", job_id,
            "--format=JobID,State,ExitCode,Elapsed,AllocTRES", "-n", "-P"],
        capture_output=True, text=True).stdout.strip()

    print("\n--- sacct summary ---")
    print(summary)

    main_state = None

    for line in summary.splitlines():
        fields = line.split("|")
        if len(fields) >= 2 and fields[0] == job_id:
            main_state = fields[1]
            break

    if main_state is None or not main_state.startswith("COMPLETED"):
        raise RuntimeError(
            f"SLURM job {job_id} finished with state {main_state}.")

    print(f"✅ Job {job_id} completed successfully.")
    return job_id

# 11. Build the Multi-GPU Job

## 🧩 Notebook process vs. training process

The Jupyter notebook prepares the experiment, but the heavy training script runs in the Slurm allocation.

`build_training_arguments(...)` converts your readable configuration dictionary into command-line settings such as:

```text
--n_embd 256 --n_head 8 --epochs 10 ...
```

The compute-node script reads those settings using `argparse`.


In [ ]:
# 🔒 RUN ONLY — command-line arguments

def build_training_arguments():
    values = ["--data_dir", project_relative(DATA_DIR), "--output_dir", project_relative(RESULTS_DIR),
        "--n_embd", str(MODEL_CONFIG["n_embd"]),
        "--n_head", str(MODEL_CONFIG["n_head"]),
        "--n_layer", str(MODEL_CONFIG["n_layer"]),
        "--dropout", str(MODEL_CONFIG["dropout"]),
        "--epochs", str(MODEL_CONFIG["epochs"]),
        "--local_batch_size", str(MODEL_CONFIG["local_batch_size"]),
        "--learning_rate", str(MODEL_CONFIG["learning_rate"]),
        "--weight_decay", str(MODEL_CONFIG["weight_decay"]),
        "--warmup_ratio", str(MODEL_CONFIG["warmup_ratio"]),
        "--precision", str(MODEL_CONFIG["precision"]),
        "--bpe_merges", str(MODEL_CONFIG["bpe_merges"]),
        "--seed", str(MODEL_CONFIG["seed"])]

    return " ".join(shlex.quote(value) for value in values)

In [ ]:
# 🔒 RUN ONLY — generate the Slurm job

def write_training_slurm(walltime="01:30:00"):
    plan = resolve_slurm(GPU_COUNT)

    arguments = build_training_arguments()
    job_name = "nb3b-custom-all-tokenizers"
    path = SCRIPTS_DIR / "custom_all_tokenizers.slurm"

    header = [
        "#!/bin/bash",
        f"#SBATCH -A {NERSC_ACCOUNT}",
        "#SBATCH -C gpu",
        f"#SBATCH -q {plan['qos']}",
        f"#SBATCH -t {walltime}",
        "",
        f"#SBATCH -N {plan['nodes']}",
        f"#SBATCH --ntasks-per-node={plan['tasks_per_node']}",
        "#SBATCH --cpus-per-task=32",
        f"#SBATCH --gpus-per-node={plan['gpus_per_node']}",
        "#SBATCH --gpu-bind=none",
        "",
        f"#SBATCH -J {job_name}",
        f"#SBATCH -o {project_relative(SLURM_LOG_DIR)}/{job_name}-%j.out",
    ]
    if plan["reservation"]:
        header.insert(4, f"#SBATCH --reservation={plan['reservation']}")

    body = f"""
export SLURM_CPU_BIND="cores"
export MASTER_ADDR=$(scontrol show hostnames "$SLURM_JOB_NODELIST" | head -n 1)
export MASTER_PORT=$((10000 + SLURM_JOB_ID % 50000))
export NCCL_DEBUG=VERSION

echo "[BATCH] CUDA_VISIBLE_DEVICES before cleanup=${{CUDA_VISIBLE_DEVICES-<unset>}}"
echo "[BATCH] SLURM_JOB_GPUS=${{SLURM_JOB_GPUS-<unset>}}"

unset CUDA_VISIBLE_DEVICES
unset NVIDIA_VISIBLE_DEVICES
unset ROCR_VISIBLE_DEVICES
unset GPU_DEVICE_ORDINAL

srun --gpu-bind=none bash -c '
    export RANK=$SLURM_PROCID
    export LOCAL_RANK=$SLURM_LOCALID
    export WORLD_SIZE=$SLURM_NTASKS

    echo "[SLURM→DDP] RANK=$RANK LOCAL_RANK=$LOCAL_RANK WORLD_SIZE=$WORLD_SIZE CUDA_VISIBLE_DEVICES=${{CUDA_VISIBLE_DEVICES-<unset>}}"

    {NOTEBOOK_PYTHON} {project_relative(TRAIN_SCRIPT)} {arguments}
'
"""

    path.write_text("\n".join(header) + "\n" + body)

    if not validate_shell_script(path):
        raise RuntimeError("Fix shell syntax before submitting.")

    return path, job_name

In [ ]:
# ✏️ RUN THIS — generate and inspect the job

TRAIN_SLURM_SCRIPT, TRAIN_JOB_NAME = (write_training_slurm(walltime="01:30:00")
)

print(TRAIN_SLURM_SCRIPT.read_text())

### ✅ CHECKPOINT — before submission

For a 2-GPU run, the printed script should contain:

```bash
#SBATCH --ntasks-per-node=2
#SBATCH --cpus-per-task=32
#SBATCH --gpus-per-node=2
#SBATCH --gpu-bind=none
```

and:

```bash
RANK=$SLURM_PROCID
LOCAL_RANK=$SLURM_LOCALID
WORLD_SIZE=$SLURM_NTASKS
```

At runtime we want:

```text
rank 0 → cuda:0
rank 1 → cuda:1
```

### Predict the cost before you spend it

This one job trains **five** models back to back, on the full dataset, on
whatever GPUs you reserved. That is a real amount of allocation, so the same
habit as Notebook 2's section 13a applies here: say what you expect before
you spend it.

Attention cost grows with the **square** of the token count, so the five runs
will not take equal time — and the cheapest one is not obviously the worst.

In [ ]:
# 👀 READ — what are you about to ask the reservation for?

plan = resolve_slurm(GPU_COUNT)
global_batch = MODEL_CONFIG["local_batch_size"] * GPU_COUNT

# Token counts per 200-bp sequence, matching the registry in Notebook 2.
# (The real registry lives inside the training script; these are the same
# numbers, kept here only so this cell can report cost before submitting.)
TOKENS_PER_SEQUENCE = {"single_nucleotide": 200, "one_hot": 200,
    "overlap_6mer": 195, "nonoverlap_6mer": 33, "bpe": 200}

print("THE JOB YOU ARE ABOUT TO SUBMIT")
print("-" * 58)
print(f"  Tokenizers (models)  : {len(TOKENIZER_ORDER)}")
print(f"  Epochs each          : {MODEL_CONFIG['epochs']}")
print(f"  GPUs                 : {GPU_COUNT} "
      f"({plan['nodes']} node(s) x {plan['gpus_per_node']})")
print(f"  Local batch / GPU    : {MODEL_CONFIG['local_batch_size']}")
print(f"  Global batch         : {global_batch}")
if plan["reservation"]:
    print(f"  Reservation          : {plan['reservation']} "
          f"({plan['window']})")
print()

print("Relative attention cost per sequence (token count squared):")
print()
cheapest = min(TOKENS_PER_SEQUENCE[n] ** 2 for n in TOKENIZER_ORDER)

for name in TOKENIZER_ORDER:
    tokens = TOKENS_PER_SEQUENCE[name]
    pairs = tokens ** 2
    print(f"  {name:<20} ~{tokens:>4} tokens  "
          f"{pairs:>8,} pairs  ({pairs / cheapest:>5.1f}x)")

print()
print("Predict before you submit:")
print("  1. Which tokenizer finishes fastest?")
print("  2. Which scores highest?")
print("  3. If those are different tokenizers, which would you deploy?")


# 12. Train All Five Tokenizers

In [ ]:
# ✏️ RUN THIS

TRAIN_JOB_ID = submit_and_stream(TRAIN_SLURM_SCRIPT, TRAIN_JOB_NAME)

### What the log should show

For each tokenizer:

```text
TOKENIZER: single_nucleotide
...
FINAL TOKENIZER REPORT

TOKENIZER: one_hot
...
FINAL TOKENIZER REPORT

TOKENIZER: overlap_6mer
...

TOKENIZER: nonoverlap_6mer
...

TOKENIZER: bpe
...
```

At the very end:

```text
ALL TOKENIZERS COMPLETED
```

The script saves each model before moving to the next one.

# 13. Load the Combined Results

In [ ]:
# 🔒 RUN ONLY

COMBINED_RESULTS_PATH = (RESULTS_DIR / "custom_all_tokenizers_summary.csv")

if not COMBINED_RESULTS_PATH.exists():
    raise FileNotFoundError("The all-tokenizer training job has not produced "
        "the combined result table yet.")

results = pd.read_csv(COMBINED_RESULTS_PATH)

results

In [ ]:
# 👀 READ — compact comparison table

comparison_columns = ["tokenizer", "mean_token_count", "total_parameters",
    "best_val_auroc", "final_val_auprc", "training_time_seconds",
    "examples_per_second", "peak_gpu_memory_gb"]

results[comparison_columns].sort_values("best_val_auroc", ascending=False)

# 14. Biological Performance Across Tokenizers

In [ ]:
plt.figure(figsize=(8, 4))

ordered = results.sort_values("best_val_auroc", ascending=False)

plt.bar(ordered["tokenizer"], ordered["best_val_auroc"])

plt.ylabel("Best validation AUROC")
plt.xlabel("Tokenizer")
plt.title("Large Custom Transformer — AUROC by Tokenizer")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))

ordered = results.sort_values("final_val_auprc", ascending=False)

plt.bar(ordered["tokenizer"], ordered["final_val_auprc"])

plt.ylabel("Final validation AUPRC")
plt.xlabel("Tokenizer")
plt.title("Large Custom Transformer — AUPRC by Tokenizer")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

# 15. Computational Cost Across Tokenizers

In [ ]:
plt.figure(figsize=(8, 4))

ordered = results.sort_values("training_time_seconds",)

plt.bar(ordered["tokenizer"], ordered["training_time_seconds"])

plt.ylabel("Training time (seconds)")
plt.xlabel("Tokenizer")
plt.title("Training Time by Tokenizer")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))

ordered = results.sort_values("examples_per_second", ascending=False)

plt.bar(ordered["tokenizer"], ordered["examples_per_second"])

plt.ylabel("Training examples / second")
plt.xlabel("Tokenizer")
plt.title("GPU Training Throughput by Tokenizer")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(results["mean_token_count"], results["training_time_seconds"],
    s=80)

for _, row in results.iterrows():
    plt.annotate(row["tokenizer"], (row["mean_token_count"],
            row["training_time_seconds"]), xytext=(5, 5),
        textcoords="offset points")

plt.xlabel("Mean token count")
plt.ylabel("Training time (seconds)")
plt.title("Token Count vs Training Time")
plt.tight_layout()
plt.show()

### What should you look for?

If tokenizer length affects compute as expected, representations with fewer tokens should often require less attention work.

But remember:

```text
shorter sequence ≠ automatically better biology
```

A tokenizer can be computationally cheap while losing useful sequence information.

The best representation should balance:

```text
predictive performance
+
computational efficiency
```

# 16. GPU Memory Across Tokenizers

In [ ]:
plt.figure(figsize=(8, 4))

ordered = results.sort_values("peak_gpu_memory_gb",)

plt.bar(ordered["tokenizer"], ordered["peak_gpu_memory_gb"])

plt.ylabel("Peak allocated GPU memory (GB)")
plt.xlabel("Tokenizer")
plt.title("Peak GPU Memory by Tokenizer")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## Why can memory differ if the architecture is the same?

The Transformer core is fixed, but tokenizers change:

- sequence length,
- attention tensor size,
- discrete vocabulary embedding size,
- dynamic-padding length.

So representation affects both **activations** and, for discrete tokenizers, part of the **parameter count**.

# 17. Model Parameter Counts

In [ ]:
plt.figure(figsize=(8, 4))

ordered = results.sort_values("total_parameters",)

plt.bar(ordered["tokenizer"], ordered["total_parameters"])

plt.ylabel("Total trainable parameters")
plt.xlabel("Tokenizer")
plt.title("Parameter Count by Tokenizer")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

### Why are the parameter counts not identical?

The **Transformer blocks are identical**.

However:

```text
single nucleotide → small embedding table
6-mer            → larger embedding table
BPE              → learned vocabulary-sized embedding table
one-hot          → linear projection instead of token embedding
```

That input layer changes the total parameter count slightly.

Students should report this rather than claiming that the models have perfectly identical parameter counts.

# 18. Performance vs Compute Tradeoff

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(results["training_time_seconds"], results["best_val_auroc"], s=80)

for _, row in results.iterrows():
    plt.annotate(row["tokenizer"], (row["training_time_seconds"],
            row["best_val_auroc"]), xytext=(5, 5), textcoords="offset points"
    )

plt.xlabel("Training time (seconds)")
plt.ylabel("Best validation AUROC")
plt.title("Performance vs Computational Cost")
plt.tight_layout()
plt.show()

## Interpreting this graph

An attractive tokenizer would move toward:

```text
higher AUROC
+
lower training time
```

There may not be one tokenizer that dominates both dimensions.

That is a real research result.

# 19. Inspect One Tokenizer in Detail

In [ ]:
# ✏️ EDIT ME

TOKENIZER_TO_INSPECT = "overlap_6mer"

HISTORY_PATH = (RESULTS_DIR
    / f"custom_large_{TOKENIZER_TO_INSPECT}_history.csv")

history = pd.read_csv(HISTORY_PATH)

history

In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(history["epoch"], history["train_loss"], marker="o",
    label="Train loss")

plt.plot(history["epoch"], history["val_loss"], marker="o",
    label="Validation loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(f"{TOKENIZER_TO_INSPECT} — Loss")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(history["epoch"], history["val_auroc"], marker="o")

plt.xlabel("Epoch")
plt.ylabel("Validation AUROC")
plt.title(f"{TOKENIZER_TO_INSPECT} — AUROC")
plt.tight_layout()
plt.show()

# 20. Student Analysis Questions

1. Which tokenizer produced the highest validation AUROC?
2. Which tokenizer produced the highest AUPRC?
3. Which tokenizer trained fastest?
4. Which tokenizer processed the most examples per second?
5. Which tokenizer used the fewest tokens per DNA sequence?
6. Did the tokenizer with the fewest tokens also train fastest?
7. Which tokenizer used the most GPU memory?
8. Why do overlapping 6-mers and non-overlapping 6-mers behave so differently computationally?
9. Why can one-hot and single-nucleotide inputs have similar token counts but different input layers?
10. Did BPE provide a useful compromise between sequence length and model performance?
11. Which tokenizer would you choose if AUROC were the only goal?
12. Which tokenizer would you choose if compute time were limited?
13. Which tokenizer gives the best overall compromise for this dataset?

# 21. The Real Question: Did From-Scratch Catch DNABERT?

This is what Notebooks 2, 3A and 3B have been building toward.

**DNABERT (Notebook 3A)** was pretrained on the human genome by someone else,
on hardware and for a length of time you do not have. You only fine-tuned it.

**Your Transformer (this notebook)** started from random weights and saw
nothing except ~1,800 CTCF sequences.

That is a very unfair fight, and the interesting part is *how* unfair it
turns out to be. The cell below puts your five tokenizers next to the
DNABERT result from Notebook 3A.

In [ ]:
# 🔒 RUN ONLY — load the Notebook 3A result for comparison

DNABERT_SUMMARY = (PROJECT_DIR / "notebook3a_results"
                   / "dnabert_maximum_full_finetune_summary.json")

dnabert_auroc = None
if DNABERT_SUMMARY.exists():
    with open(DNABERT_SUMMARY) as handle:
        dnabert_summary = json.load(handle)
    dnabert_auroc = dnabert_summary["best_val_auroc"]
    print(f"DNABERT (Notebook 3A) best validation AUROC: {dnabert_auroc:.4f}")
    print(f"  trainable parameters : "
          f"{dnabert_summary['trainable_parameters']:,}")
    print(f"  training time        : "
          f"{dnabert_summary['training_time_seconds']:.0f}s "
          f"on {dnabert_summary['num_gpus']} GPU(s)")
else:
    print("⚠️  No Notebook 3A summary found — run Notebook 3A first to compare.")
    print(f"   Looked in: {DNABERT_SUMMARY}")

print()
best_row = results.sort_values("best_val_auroc", ascending=False).iloc[0]
print(f"Best from-scratch tokenizer: {best_row['tokenizer']} "
      f"(AUROC {best_row['best_val_auroc']:.4f})")

In [ ]:
# 👀 READ — your five tokenizers against the pretrained baseline

ordered = results.sort_values("best_val_auroc")

plt.figure(figsize=(9, 4.5))
plt.barh(ordered["tokenizer"], ordered["best_val_auroc"],
         label="From scratch (this notebook)")

if dnabert_auroc is not None:
    plt.axvline(dnabert_auroc, linestyle="--", color="black",
                label=f"DNABERT pretrained ({dnabert_auroc:.3f})")

plt.axvline(0.5, linestyle=":", color="grey", label="Random guessing")
plt.xlabel("Best validation AUROC")
plt.title("From-Scratch Transformer vs Pretrained DNABERT")
plt.xlim(0.4, 1.0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

### 🧠 What this comparison is actually telling you

Whatever gap you see is the value of **pretraining**. DNABERT arrived already
knowing what human DNA looks like; your model had to learn that *and* the
CTCF task from 1,800 examples at the same time.

Some questions worth arguing about with a partner:

1. **How big is the gap?** If your best tokenizer is close, what does that
   say about how much of this task is learnable from the local sequence
   alone?
2. **Which tokenizer got closest, and is it the same one DNABERT uses?**
   DNABERT is committed to overlapping 6-mers because that is how it was
   pretrained. You are not.
3. **Count the parameters.** Compare `trainable_parameters` for both. If
   your model is smaller and close in AUROC, which one would you actually
   deploy?
4. **Compare the training times.** Yours trained from nothing in minutes.
   DNABERT's pretraining is not in that number at all. Where did that cost
   go, and who paid it?

### 🧪 Try to close the gap

The architecture knobs are yours. In order of what usually helps most on
this dataset:

```text
n_layer   : 8  -> 12
n_embd    : 256 -> 384    (keep divisible by n_head)
epochs    : 10 -> 20
```

Change **one** at a time, re-submit, and add the result to the plot above.
Record what you predicted before each run — that is the habit this whole
bootcamp is trying to build.

# 22. Notebook 3B Summary

Notebook 3B scales the custom Transformer from Notebook 2 into a multi-GPU experiment.

The workflow is:

```text
same cleaned CTCF dataset
↓
same train/validation split
↓
five DNA tokenizers
↓
five fresh large custom Transformers
↓
BF16 + DDP on every tokenizer run
↓
save five checkpoints
↓
compare biology + compute
```

The central scientific question is:

> **How does DNA representation change both predictive performance and computational cost when the Transformer architecture is held constant?**

The central HPC question is:

> **How do token count, dynamic padding, and vocabulary representation affect GPU training time, throughput, and memory?**

# 



✅ End of Notebook 3B